# Project 3, Part B: Flow Matching from Scratch

Completed, runnable implementation using the course's embedded Colab support. Select a GPU runtime, run the setup cell, then use **Runtime → Run all**. The notebook saves the required losses, sample grids, and checkpoints under `results/partb/`.

The full run trains the required 5-epoch denoisers and 10-epoch flow models. Run time depends on the Colab GPU. The final cell creates a ZIP of outputs for your webpage.

## Start here
1. Select a **GPU**: Colab **Runtime → Change runtime type**, or Kaggle **Settings → Accelerator**. Enable Internet on Kaggle.
2. Run the **Setup** cell below. Everything needed is included or downloaded automatically.
3. Complete the exercises below and run their cells.

**No ZIP upload, Google Drive mount, helper-file edits, or cache configuration is needed.**
Use a fresh runtime when switching between Part A and Part B. Downloads are automatically
reused within the session. Save your notebook with outputs before disconnecting.

In [ ]:
#@title Setup (run once)
# Supplied support code and images are embedded: no separate files are needed.
import os, sys, time, io, base64, zipfile, hashlib
from pathlib import Path
_cs180_started = time.perf_counter()
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = Path('/kaggle/working').is_dir()
PART = 'partb'
_work = Path('/content/cs180-hw3') if IN_COLAB else (Path('/kaggle/working') if IN_KAGGLE else Path.cwd())
_work.mkdir(parents=True,exist_ok=True)
_support = base64.b64decode('UEsDBBQAAAAIAAAAIVx0NVtTYAUAACkNAAAOAAAAY29sYWJfc2V0dXAucHm1V11T3DYUfd9foZIH2YkRENI23Yk7QwlJmBSyA6QvDOPR2nd3VWzJleSFLcN/75VkG8MCaTPTfVivdaVzv8/VbmxsHFxD3lgoyHRFpLIwVeqSGLBNTfjMgibG8rmQc2IXQExT10pbMm1kUQIpVc7LcsU2NjZGovKS8CjFlFVgecEt7yTKjGZaVaTmdoHydieZ4Gu3xTTTWqscjOlXVmY0yjSgEdri869GaLQ1JecXo9GogBkREmVlmVXCGDQzuuISvYnHI4IfNOwwyMlMaGMT54UkDgchCSctMlGyXJGZ0ugSL1BBzfNLPgeD+7kl+YLLORTeTwc7L9WUl2TNLi8Us3VBsMZ9NBcGyEkjrajgQGulIzrplF2BBtLUGDU0QUhULgxpI8LISWurS8S+KvkU04Q+K0loD//Ih04BHQN0VTdSukz67I4JfUUTQtmfSshozeI49phtxpx9eJKZGnIxE6BNl77TbuUUrD/R5iHFBHkAjGkbzMSddm6FFDFhoTJRfBcbq1djskRw9CldLyTWiqIWL+4PwnUO9WO1x9rQHiv7QWHN+oDf6ThWEnoQzFu7TjDqTkTQ9m4JW8OZPnQ3cv7E485jxusaZNFZ98pLu4pwx9t9g1rAPGjpX1+QQ5mXTQEYBC6NsGIJpAAHCDIXWBsRsDkjVl2CFH+jUXHiQRupoQz14r3HH5UqmhIM88ChnjOtlDXpjeQVYBJLYSPKaHy+feET5Jadd9htrD1960+H0klvbvtcBnHS9weeeiTsnTgrhLFaTBuLMcRcr+ccYxMguwgPDR6TXGGjyOYuS4N6crs7ReN7HeALKdh+3u64+E8F9T1FVXOkLXew1kJifFvicQ3X81Zvruu8vvnawmhb7o4EWb6A/DJzHBudu9yA52o+xQTQzYomtBY1frcUiL82N1ERWJq8bDEv2grsnRByyUvhGAZxER+TsdanqvTsEwL4RJtKuPpfW9ThP2xPXCM/pATNG69T7IP2i7+Ti9uscVewWH1VXQL2VIITzxMv7uelBl6sNh9OinZIjJ9n42c59xV9juWVzCH5BteH8dbIboLPuZCMnC2gm5NDi68EzsWpmwyNwQDSuB2p2HQzMW80YDA1zsxuxA75KP3ASwNtVbhmTd0gZ/lV0RaUry6/GCmsXLkUWkmGZqEG3pTYIPunO2+3s/29/U8H2fvDE5ogVUQOaosyf5rG8QCLVZeF0M4kkMhlZ7qBBK6RXzJ16d/C3ieUffqQffpydBCUeLwtumjmbqrNeO5UPXc6mDo5+XI0OQsW3wPCfq1qm0E1haJARPOv4M5O9g6PH0HDCSDcnP4GyORw8jB4nTnICnHfAI9l764HXpDPAHWoNGrI/tf3e5uu7LEBkGZw3uh8seW/lyLMxmGDINHx3LIeLUz29IZW3Nalco1Ox/TXdJf9nLx7gwQ1wZJTV37tF/aa3g7b2xVbmlL34PQ+obc3hnA1im6oH5LIThXSDYKl6Rv24w7bRQWDnG4umqkXbrPdn1BZQkFIVRuvfZu9Td7t0Ns7hnriIjngzxCNtYUQmOGk98ssbwrOhMn4kovSkfaQRB/jnlMoIXe30o+Tr4TnOb5qjliOjAMV4O/PfD5H/u8usn2vu/Z1wFiLtUkHFszBYu6XIofMyUBbLIBoOx7OKlSII8kfZe4ykNA/TvaOyEfxGy5rR8tREFqFMcoqqJRebb1++XJ3O9mJ70FNVmdON54LNmRZOxOyLKGDeHUbwttw2z24391fjEAAeMI/+8p+omA6QxyL7uEb3ti8QjJDyra7r0lk4doS7ARV4P8bLt3dyd0n8CICZdwFMtzOgurRP1BLAQIUAxQAAAAIAAAAIVx0NVtTYAUAACkNAAAOAAAAAAAAAAAAAACAAQAAAABjb2xhYl9zZXR1cC5weVBLBQYAAAAAAQABADwAAACMBQAAAAA=')
assert hashlib.sha256(_support).hexdigest() == 'ba2fae2d7267a416d94cd581fd0ac5eadf93d641618063a782d0dffdc36d31cd', 'Damaged notebook support data'
with zipfile.ZipFile(io.BytesIO(_support)) as _archive:
    for _entry in _archive.infolist():
        _target=(_work/_entry.filename).resolve()
        if _work.resolve() not in _target.parents: raise ValueError('Unsafe embedded path')
    _archive.extractall(_work)
os.chdir(_work)
if str(_work) not in sys.path: sys.path.insert(0,str(_work))
from colab_setup import configure
configure(PART,install_dependencies=IN_COLAB or IN_KAGGLE)
print('Setup complete. Continue with the notebook below.')

## Setup environment

In [ ]:
The starter’s embedded setup cell extracts the course support code and installs the matching runtime dependencies. This notebook uses that setup; no extra requirements file or local project checkout is needed in Colab.


In [ ]:
import os, json, time, random
from tqdm.auto import tqdm
QUICK_RUN = False  # Set True only for a short pipeline check
from pathlib import Path
from typing import Optional
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor
from torchvision.utils import make_grid
import matplotlib.pyplot as plt

SEED = 180
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(4)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
GROUP = os.environ.get('PARTB_GROUP','all')
assert GROUP in ('all','denoise','time','class','no_scheduler')
OUT = Path('results/partb') / GROUP
OUT.mkdir(parents=True, exist_ok=True)
metrics = {'seed':SEED,'group':GROUP,'torch':torch.__version__,
           'device':str(device),'gpu':torch.cuda.get_device_name() if device.type=='cuda' else None,
           'visible_devices':os.environ.get('CUDA_VISIBLE_DEVICES'),
           'precision':'float32 with TF32 convolution/matmul enabled'}
print(metrics)
train_dataset = MNIST('data', train=True, download=True, transform=ToTensor())
test_dataset = MNIST('data', train=False, download=True, transform=ToTensor())
assert len(train_dataset)==60000 and len(test_dataset)==10000

def show_grid(images, title, filename, nrow=10):
    images = images.detach().float().cpu()
    grid = make_grid(images.clamp(0, 1), nrow=nrow, padding=2)
    fig, ax = plt.subplots(figsize=(12, max(2, len(images)/nrow*1.15)))
    ax.imshow(grid.permute(1, 2, 0)); ax.set_title(title); ax.axis('off')
    fig.tight_layout(); fig.savefig(OUT / filename, dpi=120)
    plt.show(); plt.close(fig)


def loss_plot(losses, name):
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(losses, alpha=.4, linewidth=.5)
    window = min(100, len(losses))
    smooth = torch.tensor(losses).unfold(0, window, 1).mean(1)
    ax.plot(range(window-1, len(losses)), smooth, linewidth=1)
    ax.set(xlabel='Optimizer step', ylabel='Mean squared error', title=name)
    fig.tight_layout(); fig.savefig(OUT / (name+'_loss.png'), dpi=120)
    plt.show(); plt.close(fig)


def make_loader(batch_size, seed=180):
    # Decode/normalize once; augmentation noise is still drawn fresh in each step.
    dataset = torch.utils.data.TensorDataset(train_dataset.data[:, None].float()/255,
                                             train_dataset.targets)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0,
                      pin_memory=device.type == 'cuda',
                      generator=torch.Generator().manual_seed(seed))

## Neural Network Resources

In this part, you will build and train a [UNet](https://arxiv.org/abs/1505.04597), which is more complex than the MLP you implemented in the NeRF project.  We provide all class definitions you may need (but feel free to add or modify them as necessary).  

Instead of asking ChatGPT to write everything for you, please consult the following resources when you get stuck — they will help you understand how and why things work under the hood.

- PyTorch Documentation — [`Conv2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html), [`ConvTranspose2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.ConvTranspose2d.html), and [`AvgPool2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.AvgPool2d.html).
- PyTorch Documentation - [`torchvision.datasets.MNIST`](https://docs.pytorch.org/vision/main/generated/torchvision.datasets.MNIST.html), the dataset we gonna use, and [`torch.utils.data.DataLoader`](https://docs.pytorch.org/docs/stable/data.html), the off-the-shell dataloader we can directly use.
- PyTorch [tutorial](https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html) on how to train a classifier on CIFAR10 dataset. The structure of your training code will be very similar to this one.

# Part 1: Training a Single-step Denoising UNet


# Part 1.1: Implementing the UNet

## Implementing Simple and Composed Ops

In [ ]:
class Conv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn = nn.BatchNorm2d(out_channels)
        self.activation = nn.GELU()
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation(self.bn(self.conv(x)))

class DownConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=2, padding=1)
        self.bn = nn.BatchNorm2d(out_channels)
        self.activation = nn.GELU()
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation(self.bn(self.conv(x)))

class UpConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.conv = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1)
        self.bn = nn.BatchNorm2d(out_channels)
        self.activation = nn.GELU()
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation(self.bn(self.conv(x)))

class Flatten(nn.Module):
    def __init__(self):
        super().__init__(); self.pool = nn.AvgPool2d(kernel_size=7); self.activation = nn.GELU()
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation(self.pool(x))

class Unflatten(nn.Module):
    def __init__(self, in_channels: int):
        super().__init__()
        self.conv = nn.ConvTranspose2d(in_channels, in_channels, kernel_size=7, stride=7)
        self.bn = nn.BatchNorm2d(in_channels); self.activation = nn.GELU()
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation(self.bn(self.conv(x)))

class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__(); self.layers = nn.Sequential(Conv(in_channels,out_channels), Conv(out_channels,out_channels))
    def forward(self, x: torch.Tensor) -> torch.Tensor: return self.layers(x)

class DownBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__(); self.layers = nn.Sequential(DownConv(in_channels,out_channels), ConvBlock(out_channels,out_channels))
    def forward(self, x: torch.Tensor) -> torch.Tensor: return self.layers(x)

class UpBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__(); self.up = UpConv(in_channels,out_channels); self.conv = ConvBlock(out_channels,out_channels)
    def forward(self, x: torch.Tensor) -> torch.Tensor: return self.conv(self.up(x))

## Implementing Unconditional UNet

In [ ]:
class UnconditionalUNet(nn.Module):
    def __init__(self, in_channels: int, num_hiddens: int):
        super().__init__(); d=num_hiddens
        self.conv = Conv(in_channels,d)
        self.down1 = DownBlock(d,d)
        self.down2 = DownBlock(d,2*d)
        self.flatten = Flatten()
        self.unflatten = Unflatten(2*d)
        self.up1 = UpBlock(4*d,2*d)
        self.up2 = UpBlock(3*d,d)
        self.out = Conv(d,in_channels)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        assert x.shape[-2:] == (28,28), "Expect input shape (28,28)."
        x1=self.conv(x); x2=self.down1(x1); x3=self.down2(x2)
        x4=self.flatten(x3); x5=self.unflatten(x4)
        x6=self.up1(torch.cat([x5,x3],dim=1))
        x7=self.up2(torch.cat([x6,x2],dim=1))
        return self.out(x7)

# Part 1.2: Using the UNet to Train a Denoiser

In [ ]:
# Visualize additive Gaussian noise at all requested levels, using a fixed clean digit.
sigmas = [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]
sample, _ = test_dataset[0]
sample = sample.unsqueeze(0).to(device)
eps = torch.randn_like(sample)
noisy_rows = [sample] + [(sample + s*eps).clamp(0,1) for s in sigmas]
show_grid(torch.cat(noisy_rows), 'Clean digit followed by additive noise σ = 0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0', 'noise_process.png', nrow=8)

## Part 1.2.1: Training

For this part, we provide some structure code for training. It is very basic, so feel free to change them or add your code. In later section we won't provide any training or visualization structure code.

In [ ]:
batch_size = 256
learning_rate = 1e-4
noise_level = 0.5
hidden_dim = 128
num_epochs = 1 if QUICK_RUN else 5

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=device.type=='cuda')
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2, pin_memory=device.type=='cuda')

In [ ]:
model = UnconditionalUNet(in_channels=1, num_hiddens=hidden_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.MSELoss()

In [ ]:
train_losses = []
epoch_outputs = {}
for epoch in range(num_epochs):
    model.train()
    for i, (images, _) in enumerate(tqdm(train_loader)):
        images = images.to(device, non_blocking=True)
        noisy_images = images + noise_level * torch.randn_like(images)
        outputs = model(noisy_images)
        loss = criterion(outputs, images)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
    model.eval()
    clean, _ = next(iter(test_loader)); clean=clean[:64].to(device)
    noisy=clean+noise_level*torch.randn_like(clean)
    with torch.inference_mode(): pred=model(noisy).clamp(0,1)
    epoch_outputs[epoch+1]=(noisy.cpu(),pred.cpu(),clean.cpu())
    print(f"Epoch {epoch+1}/{num_epochs}: loss={train_losses[-1]:.5f}")
    if epoch+1 in (1,5):
        show_grid(torch.cat([noisy,pred,clean]), f"Denoiser epoch {epoch+1} · noisy / output / clean", f"denoiser_epoch_{epoch+1}.png", nrow=8)
torch.save(model.state_dict(), Path('results/partb/denoiser_sigma05.pt'))

In [ ]:
fig, ax = plt.subplots(figsize=(8,3))
ax.plot(train_losses, linewidth=.6)
ax.set(xlabel='Training step', ylabel='MSE', title='Single-step denoiser loss (σ=0.5)')
fig.tight_layout(); fig.savefig('results/partb/denoiser_loss.png',dpi=140); plt.show(); plt.close(fig)

## Part 1.2.2: Out-of-Distribution Testing

In [ ]:
# Evaluate one fixed test digit at every requested noise level.
model.eval(); fixed,_=test_dataset[0]; fixed=fixed.unsqueeze(0).to(device)
fixed_eps=torch.randn_like(fixed)
noisy=[]; restored=[]
with torch.inference_mode():
    for sigma in sigmas:
        z=fixed+sigma*fixed_eps; noisy.append(z); restored.append(model(z).clamp(0,1))
show_grid(torch.cat([fixed.repeat(len(sigmas),1,1,1),*noisy,*restored]),
          'OOD test · rows: clean reference, noisy at σ, denoised at σ', 'denoiser_ood.png', nrow=len(sigmas))

## Part 1.2.3 Denoising Pure Noise

In [ ]:
# Pure-noise denoising: fresh independent Gaussian input, clean target.
pure_model=UnconditionalUNet(1,hidden_dim).to(device)
pure_optimizer=torch.optim.Adam(pure_model.parameters(),lr=1e-4)
pure_losses=[]
for epoch in range(num_epochs):
    pure_model.train()
    for images,_ in tqdm(train_loader):
        images=images.to(device,non_blocking=True); z=torch.randn_like(images)
        pred=pure_model(z); loss=F.mse_loss(pred,images)
        pure_optimizer.zero_grad(set_to_none=True); loss.backward(); pure_optimizer.step()
        pure_losses.append(loss.item())
    pure_model.eval(); clean,_=next(iter(test_loader)); clean=clean[:64].to(device); z=torch.randn_like(clean)
    with torch.inference_mode(): out=pure_model(z).clamp(0,1)
    if epoch+1 in (1,5): show_grid(torch.cat([z.clamp(0,1),out,clean]),f'Pure-noise denoiser epoch {epoch+1} · noise / output / clean',f'pure_noise_epoch_{epoch+1}.png',nrow=8)
fig,ax=plt.subplots(figsize=(8,3)); ax.plot(pure_losses,linewidth=.6); ax.set(xlabel='Step',ylabel='MSE',title='Pure-noise denoiser loss'); fig.tight_layout(); fig.savefig('results/partb/pure_noise_loss.png',dpi=140); plt.show(); plt.close(fig)
torch.save(pure_model.state_dict(),Path('results/partb/pure_noise_denoiser.pt'))
print('With MSE and input independent of the target, the optimal prediction is the pixelwise mean of the training digits. It resembles a blurred average digit rather than a sharp sample.')

# Part 2: Flow Matching

# Part 2.1: Implementing a Time-conditioned UNet

In [ ]:
class FCBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__(); self.layers=nn.Sequential(nn.Linear(in_channels,out_channels),nn.GELU(),nn.Linear(out_channels,out_channels))
    def forward(self, x: torch.Tensor) -> torch.Tensor: return self.layers(x)

class TimeConditionalUNet(nn.Module):
    def __init__(self,in_channels:int,num_classes:int,num_hiddens:int):
        super().__init__(); d=num_hiddens
        self.conv=Conv(in_channels,d); self.down1=DownBlock(d,d); self.down2=DownBlock(d,2*d)
        self.flatten=Flatten(); self.unflatten=Unflatten(2*d)
        self.up1=UpBlock(4*d,2*d); self.up2=UpBlock(3*d,d); self.out=Conv(d,in_channels)
        self.fc1_t=FCBlock(1,2*d); self.fc2_t=FCBlock(1,d)
    def forward(self,x:torch.Tensor,t:torch.Tensor)->torch.Tensor:
        assert x.shape[-2:]==(28,28), "Expect input shape (28,28)."
        t=t.to(x).reshape(-1,1).clamp(0,1)
        x1=self.conv(x); x2=self.down1(x1); x3=self.down2(x2)
        x4=self.flatten(x3); x5=self.unflatten(x4)
        t1=self.fc1_t(t)[:,:,None,None]; t2=self.fc2_t(t)[:,:,None,None]
        x5=x5*t1
        x6=self.up1(torch.cat([x5,x3],dim=1)); x6=x6*t2
        x7=self.up2(torch.cat([x6,x2],dim=1))
        return self.out(x7)

## Implementing the Forward and Reverse Process for Time-conditioned Denoising

In [ ]:
def time_fm_forward(unet:TimeConditionalUNet,x_1:torch.Tensor,num_ts:int)->torch.Tensor:
    """Sample a linear OT point and regress its constant target velocity."""
    unet.train(); x_1=x_1.to(device)
    x_0=torch.randn_like(x_1); t=torch.rand(x_1.shape[0],device=x_1.device)
    t_img=t[:,None,None,None]
    x_t=(1-t_img)*x_0+t_img*x_1
    return F.mse_loss(unet(x_t,t),x_1-x_0)

In [ ]:
@torch.inference_mode()
def time_fm_sample(unet:TimeConditionalUNet,img_wh:tuple[int,int],num_ts:int,seed:int=0)->torch.Tensor:
    """Euler integrate a batch of 64 standard-normal images from t=0 to t=1."""
    unet.eval(); h,w=img_wh
    gen=torch.Generator(device=device).manual_seed(seed)
    x=torch.randn(64,1,h,w,device=device,generator=gen)
    for i in range(num_ts):
        t=torch.full((len(x),),i/num_ts,device=device)
        x=x+(1.0/num_ts)*unet(x,t)
    return x.clamp(0,1)

In [ ]:
class TimeConditionalFM(nn.Module):
    def __init__(
        self,
        unet: TimeConditionalUNet,
        num_ts: int = 50,
        img_hw: tuple[int, int] = (28, 28),
    ):
        super().__init__()

        self.unet = unet
        self.num_ts = num_ts
        self.img_hw = img_hw


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, C, H, W) input tensor.

        Returns:
            (,) flow-matching loss.
        """
        return time_fm_forward(
            self.unet, x, self.num_ts
        )


    @torch.inference_mode()
    def sample(
        self,
        img_wh: tuple[int, int],
        seed: int = 0,
    ):
        return time_fm_sample(
            self.unet, img_wh, self.num_ts, seed
        )

# Part 2.2: Training the Time-conditioned UNet

In [ ]:
# Train the time-only model with the assignment's exponential learning-rate decay.
flow_epochs=1 if QUICK_RUN else 10
flow_dim=16 if QUICK_RUN else 64
flow_batch=64
num_ts=100 if QUICK_RUN else 100
flow_loader=DataLoader(train_dataset,batch_size=flow_batch,shuffle=True,num_workers=2,pin_memory=device.type=='cuda')
time_unet=TimeConditionalUNet(1,10,flow_dim).to(device)
time_opt=torch.optim.Adam(time_unet.parameters(),lr=1e-2)
time_sched=torch.optim.lr_scheduler.ExponentialLR(time_opt,gamma=0.1**(1/flow_epochs))
time_losses=[]; time_checkpoints={}
for epoch in range(flow_epochs):
    for images,_ in tqdm(flow_loader):
        images=images.to(device,non_blocking=True)
        loss=time_fm_forward(time_unet,images,num_ts)
        time_opt.zero_grad(set_to_none=True); loss.backward(); time_opt.step(); time_losses.append(loss.item())
    time_sched.step(); print(f'Time flow epoch {epoch+1}/{flow_epochs}: loss={time_losses[-1]:.5f}')
    if epoch+1 in (1,5,10) or QUICK_RUN:
        time_checkpoints[epoch+1]={k:v.detach().cpu().clone() for k,v in time_unet.state_dict().items()}
        torch.save(time_unet.state_dict(),f'results/partb/time_flow_epoch_{epoch+1}.pt')
fig,ax=plt.subplots(figsize=(8,3)); ax.plot(time_losses,linewidth=.6); ax.set(xlabel='Step',ylabel='MSE',title='Time-conditioned flow loss'); fig.tight_layout(); fig.savefig('results/partb/time_flow_loss.png',dpi=140); plt.show(); plt.close(fig)

# Part 2.3: Sampling from the Time-conditioned UNet

In [ ]:
# Generate from the saved epoch checkpoints, with fixed seeds for comparison.
time_samples={}
for epoch,state in time_checkpoints.items():
    time_unet.load_state_dict(state); samples=time_fm_sample(time_unet,(28,28),num_ts=300 if not QUICK_RUN else 30,seed=SEED+epoch)
    time_samples[epoch]=samples
    show_grid(samples,f'Time-conditioned flow samples · epoch {epoch}',f'time_flow_samples_epoch_{epoch}.png',nrow=8)

# Part 2.4: Implementing a Class-conditioned UNet

In [ ]:
class ClassConditionalUNet(nn.Module):
    def __init__(self,in_channels:int,num_classes:int,num_hiddens:int):
        super().__init__(); d=num_hiddens; self.num_classes=num_classes
        self.conv=Conv(in_channels,d); self.down1=DownBlock(d,d); self.down2=DownBlock(d,2*d)
        self.flatten=Flatten(); self.unflatten=Unflatten(2*d)
        self.up1=UpBlock(4*d,2*d); self.up2=UpBlock(3*d,d); self.out=Conv(d,in_channels)
        self.fc1_t=FCBlock(1,2*d); self.fc2_t=FCBlock(1,d)
        self.fc1_c=FCBlock(num_classes,2*d); self.fc2_c=FCBlock(num_classes,d)
    def forward(self,x:torch.Tensor,c:torch.Tensor,t:torch.Tensor,mask:Optional[torch.Tensor]=None)->torch.Tensor:
        assert x.shape[-2:]==(28,28), "Expect input shape (28,28)."
        t=t.to(x).reshape(-1,1).clamp(0,1); c=c.to(device=x.device,dtype=torch.long).reshape(-1)
        if mask is None: mask=(c>=0).to(x.dtype)
        else: mask=mask.to(device=x.device,dtype=x.dtype).reshape(-1)
        c_safe=c.clamp(min=0,max=self.num_classes-1)
        c_onehot=F.one_hot(c_safe,num_classes=self.num_classes).to(x.dtype)*mask[:,None]
        t1=self.fc1_t(t)[:,:,None,None]; t2=self.fc2_t(t)[:,:,None,None]
        c1=self.fc1_c(c_onehot)[:,:,None,None]; c2=self.fc2_c(c_onehot)[:,:,None,None]
        x1=self.conv(x); x2=self.down1(x1); x3=self.down2(x2)
        x4=self.flatten(x3); x5=self.unflatten(x4)
        x5=c1*x5+t1
        x6=self.up1(torch.cat([x5,x3],dim=1)); x6=c2*x6+t2
        x7=self.up2(torch.cat([x6,x2],dim=1))
        return self.out(x7)

In [ ]:
def class_fm_forward(unet:ClassConditionalUNet,x_1:torch.Tensor,c:torch.Tensor,p_uncond:float,num_ts:int)->torch.Tensor:
    """Flow-matching objective with random class-condition dropout."""
    unet.train(); x_1=x_1.to(device); c=c.to(device).long()
    x_0=torch.randn_like(x_1); t=torch.rand(len(x_1),device=device)
    x_t=(1-t[:,None,None,None])*x_0+t[:,None,None,None]*x_1
    mask=(torch.rand(len(c),device=device)>=p_uncond).to(x_1.dtype)
    return F.mse_loss(unet(x_t,c,t,mask),x_1-x_0)

In [ ]:
@torch.inference_mode()
def class_fm_sample(unet:ClassConditionalUNet,c:torch.Tensor,img_wh:tuple[int,int],num_ts:int,guidance_scale:float=5.0,seed:int=0)->torch.Tensor:
    """Sample with Part B CFG convention v_u + γ(v_c-v_u)."""
    unet.eval(); c=c.to(device).long().reshape(-1); h,w=img_wh
    gen=torch.Generator(device=device).manual_seed(seed)
    x=torch.randn(len(c),1,h,w,device=device,generator=gen)
    null=torch.zeros_like(c); mask_c=torch.ones(len(c),device=device); mask_u=torch.zeros(len(c),device=device)
    for i in range(num_ts):
        t=torch.full((len(c),),i/num_ts,device=device)
        v_c=unet(x,c,t,mask_c); v_u=unet(x,c,t,mask_u)
        v=v_u+guidance_scale*(v_c-v_u)
        x=x+(1.0/num_ts)*v
    return x.clamp(0,1)

In [ ]:
class ClassConditionalFM(nn.Module):
    def __init__(
        self,
        unet: ClassConditionalUNet,
        num_ts: int = 300,
        p_uncond: float = 0.1,
    ):
        super().__init__()
        self.unet = unet
        self.num_ts = num_ts
        self.p_uncond = p_uncond

    def forward(self, x: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, C, H, W) input tensor.
            c: (N,) int64 condition tensor.

        Returns:
            (,) loss.
        """
        return class_fm_forward(
            self.unet, x, c, self.p_uncond, self.num_ts
        )

    @torch.inference_mode()
    def sample(
        self,
        c: torch.Tensor,
        img_wh: tuple[int, int],
        guidance_scale: float = 5.0,
        seed: int = 0,
    ):
        return class_fm_sample(
            self.unet, c, img_wh, self.num_ts, guidance_scale, seed
        )

# Part 2.5 Training the Class-conditioned UNet

In [ ]:
# Train class-conditioned models with and without the exponential scheduler.
def train_class_model(use_scheduler=True, tag='class_flow'):
    net=ClassConditionalUNet(1,10,flow_dim).to(device)
    opt=torch.optim.Adam(net.parameters(),lr=1e-2 if use_scheduler else 3e-3)
    sched=torch.optim.lr_scheduler.ExponentialLR(opt,gamma=0.1**(1/flow_epochs)) if use_scheduler else None
    losses=[]; snapshots={}
    for epoch in range(flow_epochs):
        for images,labels in tqdm(flow_loader):
            images=images.to(device,non_blocking=True); labels=labels.to(device,non_blocking=True)
            loss=class_fm_forward(net,images,labels,p_uncond=.1,num_ts=num_ts)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); losses.append(loss.item())
        if sched: sched.step()
        print(f'{tag} epoch {epoch+1}/{flow_epochs}: loss={losses[-1]:.5f}, lr={opt.param_groups[0]["lr"]:.5g}')
        if epoch+1 in (1,5,10) or QUICK_RUN:
            snapshots[epoch+1]={k:v.detach().cpu().clone() for k,v in net.state_dict().items()}
            torch.save(net.state_dict(),f'results/partb/{tag}_epoch_{epoch+1}.pt')
    return net,losses,snapshots

class_unet,class_losses,class_checkpoints=train_class_model(True,'class_flow')
fig,ax=plt.subplots(figsize=(8,3)); ax.plot(class_losses,linewidth=.6); ax.set(xlabel='Step',ylabel='MSE',title='Class-conditioned flow loss'); fig.tight_layout(); fig.savefig('results/partb/class_flow_loss.png',dpi=140); plt.show(); plt.close(fig)
no_sched_unet,no_sched_losses,no_sched_checkpoints=train_class_model(False,'class_flow_no_scheduler')
fig,ax=plt.subplots(figsize=(8,3)); ax.plot(class_losses,label='exponential schedule',linewidth=.6); ax.plot(no_sched_losses,label='constant lr = 0.003',linewidth=.6); ax.set(xlabel='Step',ylabel='MSE',title='Scheduler comparison'); ax.legend(); fig.tight_layout(); fig.savefig('results/partb/scheduler_comparison.png',dpi=140); plt.show(); plt.close(fig)

# Part 2.6: Sampling from the Class-conditioned UNet

In [ ]:
# Four samples of each digit at epochs 1, 5, 10; same initial noise seed across checkpoints.
class_samples={}
labels=torch.arange(10,device=device).repeat_interleave(4)
for epoch,state in class_checkpoints.items():
    class_unet.load_state_dict(state)
    samples=class_fm_sample(class_unet,labels,(28,28),300 if not QUICK_RUN else 30,guidance_scale=5.0,seed=SEED+100)
    class_samples[epoch]=samples
    show_grid(samples,f'Class-conditioned samples · epoch {epoch} · 4 per digit · γ=5',f'class_flow_samples_epoch_{epoch}.png',nrow=10)
for epoch,state in no_sched_checkpoints.items():
    no_sched_unet.load_state_dict(state)
    samples=class_fm_sample(no_sched_unet,labels,(28,28),300 if not QUICK_RUN else 30,guidance_scale=5.0,seed=SEED+100)
    show_grid(samples,f'No-scheduler class flow · epoch {epoch}',f'class_flow_no_scheduler_epoch_{epoch}.png',nrow=10)
print('Comparison note: the constant 0.003 learning rate removes schedule bookkeeping and is near the middle of the scheduled run’s learning-rate range. Compare the epoch-10 grids and the loss curves; keep the scheduler for the assignment baseline.')

## Export figures and checkpoints for the project website
The following cell packages all saved outputs so they can be merged with Part A assets.


In [ ]:
import zipfile
result_dir = Path('results/partb')
archive = Path('partb-website-results.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in result_dir.rglob('*'):
        if f.is_file() and (f.suffix.lower() == '.png' or f.name.endswith('.json')): zf.write(f, f.relative_to(result_dir))
print('Website figures/checkpoints:', archive)
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(str(archive))
